# D34 | PM | Take-Home | PCA & Week 6 Comprehensive Review
**IIT Gandhinagar | PG Diploma in AI-ML & Agentic AI Engineering**

Topics: PCA, All 12 Week-6 Algorithms, Image Compression, Pipeline Design, Model Comparison

---

## Global Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Datasets
from sklearn.datasets import load_wine, load_iris, load_breast_cancer

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# Model selection
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

# Supervised algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier, GradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
import lightgbm as lgb

# Unsupervised algorithms
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'DejaVu Sans'
})
PALETTE = ['#E63946', '#457B9D', '#2A9D8F', '#F4A261', '#A8DADC', '#1D3557']
print('All libraries loaded ✓')
print(f'XGBoost {xgb.__version__} | LightGBM {lgb.__version__}')

---
# PART A — Week 6 Algorithm Quick Reference (40%)
### 13 Algorithms/Techniques: 1-line description · When to use · Code snippet · Key hyperparameters

## Task 1 & 2 — Algorithm Quick Reference

---
### 1. Logistic Regression (LR)
**Description:** Linear model that uses the sigmoid function to output class probabilities; best for linearly separable data.

**When to use:** Binary or multiclass classification, when features are roughly linear, when interpretability matters, as a fast baseline.

**Key Hyperparameters:**
- `C` (float, default=1.0): Inverse of regularization strength. Smaller = more regularization.
- `solver` ('lbfgs','liblinear','saga'): Optimization algorithm. 'lbfgs' for small/medium data; 'saga' for large.

**Use Case:** Credit risk scoring — predict default (0/1) from financial features.

In [ ]:
# ── Logistic Regression Example ──────────────────────────────────────────────
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

wine = load_wine()
X_w, y_w = wine.data, wine.target
X_tr, X_te, y_tr, y_te = train_test_split(X_w, y_w, test_size=0.2, random_state=42, stratify=y_w)
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)

lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_tr_s, y_tr)
print(f'LR Accuracy: {accuracy_score(y_te, lr.predict(X_te_s)):.4f}')

---
### 2. Decision Tree (DT)
**Description:** Hierarchical if-else rules that split data by the feature which maximizes information gain (or Gini reduction).

**When to use:** When interpretability is critical, categorical features are present, or as a building block for ensembles.

**Key Hyperparameters:**
- `max_depth` (int): Maximum tree depth. Prevents overfitting.
- `min_samples_split` (int): Minimum samples required to split a node.

**Use Case:** Medical diagnosis — rule-based classification of symptoms.

In [ ]:
# ── Decision Tree Example ─────────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5, min_samples_split=4, random_state=42)
dt.fit(X_tr, y_tr)  # DT doesn't need scaling
print(f'DT Accuracy: {accuracy_score(y_te, dt.predict(X_te)):.4f}')
print(f'Tree depth used: {dt.get_depth()}')

---
### 3. Random Forest (RF)
**Description:** Ensemble of decision trees trained on bootstrap samples with random feature subsets; aggregates predictions by majority vote.

**When to use:** General-purpose classification/regression, when single DT overfits, when feature importance is needed.

**Key Hyperparameters:**
- `n_estimators` (int, default=100): Number of trees. More = better but slower.
- `max_features` ('sqrt','log2', float): Features considered at each split.

**Use Case:** Customer churn prediction — robust to noisy features.

In [ ]:
# ── Random Forest Example ─────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_features='sqrt', random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
print(f'RF Accuracy: {accuracy_score(y_te, rf.predict(X_te)):.4f}')

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=wine.feature_names).sort_values(ascending=False)
print(f'Top 3 features: {list(feat_imp.head(3).index)}')

---
### 4. AdaBoost
**Description:** Iteratively trains weak learners (stumps) on reweighted samples — misclassified samples get higher weight each round.

**When to use:** When weak learners are available, balanced datasets, binary classification tasks with low noise.

**Key Hyperparameters:**
- `n_estimators` (int): Number of boosting rounds.
- `learning_rate` (float): Shrinks each estimator's contribution. Trade-off with n_estimators.

**Use Case:** Face detection (Viola-Jones) — combining many weak features.

In [ ]:
# ── AdaBoost Example ──────────────────────────────────────────────────────────
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # weak learner = stump
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)
ada.fit(X_tr, y_tr)
print(f'AdaBoost Accuracy: {accuracy_score(y_te, ada.predict(X_te)):.4f}')

---
### 5. XGBoost
**Description:** Gradient boosted trees with regularization (L1/L2), second-order gradients, and parallel column block scanning for speed.

**When to use:** Tabular data competitions, when RF/AdaBoost plateaus, when you need high accuracy with reasonable speed.

**Key Hyperparameters:**
- `n_estimators` + `learning_rate`: Boosting rounds and shrinkage (tune together).
- `max_depth` + `subsample`: Tree complexity and stochastic row sampling.

**Use Case:** Kaggle tabular competitions — consistently top-performing.

In [ ]:
# ── XGBoost Example ───────────────────────────────────────────────────────────
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)
xgb_clf.fit(X_tr_s, y_tr)
print(f'XGBoost Accuracy: {accuracy_score(y_te, xgb_clf.predict(X_te_s)):.4f}')

---
### 6. LightGBM
**Description:** Gradient boosted trees using leaf-wise growth (vs. level-wise in XGBoost) with histogram-based splits — extremely fast on large data.

**When to use:** Large datasets (>100K rows), high-dimensional features, when XGBoost is too slow.

**Key Hyperparameters:**
- `num_leaves` (int, default=31): Controls model complexity (key LightGBM param).
- `min_child_samples` (int): Minimum data in a leaf — primary overfitting control.

**Use Case:** Real-time click-through rate (CTR) prediction at scale.

In [ ]:
# ── LightGBM Example ──────────────────────────────────────────────────────────
import lightgbm as lgb

lgb_clf = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=10,
    random_state=42,
    verbose=-1
)
lgb_clf.fit(X_tr_s, y_tr)
print(f'LightGBM Accuracy: {accuracy_score(y_te, lgb_clf.predict(X_te_s)):.4f}')

---
### 7. Voting Classifier (Ensemble)
**Description:** Combines predictions from multiple diverse models by majority vote (hard) or averaged probabilities (soft).

**When to use:** When individual models have similar accuracy but make different errors; soft voting when all estimators support predict_proba.

**Key Hyperparameters:**
- `voting` ('hard'/'soft'): Hard = majority class; soft = averaged probabilities.
- `weights` (list): Weight each model's vote differently.

**Use Case:** Medical imaging — ensemble of radiology AI models.

In [ ]:
# ── Voting Classifier Example ─────────────────────────────────────────────────
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ('lr',  LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
        ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svm', SVC(probability=True, random_state=42))
    ],
    voting='soft'  # Uses predict_proba — needs probability=True for SVC
)
voting_clf.fit(X_tr_s, y_tr)
print(f'Voting Classifier Accuracy: {accuracy_score(y_te, voting_clf.predict(X_te_s)):.4f}')

---
### 8. Stacking Classifier (Ensemble)
**Description:** Trains base learners (level-0), then a meta-learner (level-1) is trained on out-of-fold predictions from base learners.

**When to use:** When you want to capture complementary patterns from diverse base models; when overfitting of Voting is a concern.

**Key Hyperparameters:**
- `estimators` (list): Base models (level-0 learners).
- `final_estimator`: Meta-learner (level-1); default is LogisticRegression.

**Use Case:** Netflix prize — stacking diverse recommendation models.

In [ ]:
# ── Stacking Classifier Example ───────────────────────────────────────────────
from sklearn.ensemble import StackingClassifier

stacking_clf = StackingClassifier(
    estimators=[
        ('dt',  DecisionTreeClassifier(max_depth=4, random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=7)),
        ('rf',  RandomForestClassifier(n_estimators=50, random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000),  # meta-learner
    cv=5,
    passthrough=False
)
stacking_clf.fit(X_tr_s, y_tr)
print(f'Stacking Accuracy: {accuracy_score(y_te, stacking_clf.predict(X_te_s)):.4f}')

---
### 9. Support Vector Machine (SVM)
**Description:** Finds the maximum-margin hyperplane separating classes; uses the kernel trick to handle non-linear boundaries.

**When to use:** High-dimensional data (text, genomics), small-to-medium datasets, when clear margin of separation exists.

**Key Hyperparameters:**
- `C` (float): Regularization. High C = fit training data closely (less margin).
- `kernel` ('rbf','linear','poly'): Kernel function. 'rbf' is general-purpose.

**Use Case:** Text classification — separating spam from ham with TF-IDF features.

In [ ]:
# ── SVM Example ───────────────────────────────────────────────────────────────
from sklearn.svm import SVC

svm = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=42)
svm.fit(X_tr_s, y_tr)
print(f'SVM Accuracy: {accuracy_score(y_te, svm.predict(X_te_s)):.4f}')
print(f'Number of support vectors per class: {svm.n_support_}')

---
### 10. K-Nearest Neighbors (KNN)
**Description:** Classifies a new point by majority vote of its K nearest neighbours in feature space (lazy learner — no training phase).

**When to use:** Small datasets, when decision boundary is complex/irregular, as a baseline, anomaly detection.

**Key Hyperparameters:**
- `n_neighbors` (K): Number of neighbours. Odd values avoid ties for binary.
- `metric` ('euclidean','manhattan','minkowski'): Distance metric.

**Use Case:** Recommender systems — find K most similar users/items.

In [ ]:
# ── KNN Example ───────────────────────────────────────────────────────────────
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=7, metric='euclidean', weights='uniform')
knn.fit(X_tr_s, y_tr)  # KNN MUST use scaled features
print(f'KNN (K=7) Accuracy: {accuracy_score(y_te, knn.predict(X_te_s)):.4f}')

# Find best K
k_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X_tr_s, y_tr, cv=5).mean()
            for k in range(1, 16)]
print(f'Best K: {np.argmax(k_scores)+1} | CV Score: {max(k_scores):.4f}')

---
### 11. K-Means
**Description:** Partitions data into K clusters by iteratively assigning points to the nearest centroid and updating centroid positions.

**When to use:** Customer segmentation, document clustering, image quantization — when K is known or can be estimated via elbow/silhouette.

**Key Hyperparameters:**
- `n_clusters` (K): Number of clusters. Choose via elbow method or domain knowledge.
- `init` ('k-means++' / 'random'): Centroid initialization. k-means++ is almost always better.

**Use Case:** Market segmentation — grouping customers by purchasing behaviour.

In [ ]:
# ── K-Means Example ───────────────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_wine_scaled = StandardScaler().fit_transform(X_w)

km = KMeans(n_clusters=3, init='k-means++', n_init=20, random_state=42)
km_labels = km.fit_predict(X_wine_scaled)
sil = silhouette_score(X_wine_scaled, km_labels)
ari = adjusted_rand_score(y_w, km_labels)
print(f'K-Means: Silhouette={sil:.4f} | ARI={ari:.4f} | Inertia={km.inertia_:.2f}')

---
### 12. DBSCAN
**Description:** Density-based clustering that groups core points (with ≥ min_samples neighbours within eps) and marks outliers as noise (-1).

**When to use:** Arbitrary-shaped clusters, when outlier detection matters, when K is unknown, geospatial data.

**Key Hyperparameters:**
- `eps` (float): Neighbourhood radius. Tune via k-distance elbow plot.
- `min_samples` (int): Minimum neighbours to be a core point.

**Use Case:** Fraud detection — fraudulent transactions form small dense clusters separate from normal.

In [ ]:
# ── DBSCAN Example ────────────────────────────────────────────────────────────
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=1.5, min_samples=5)
db_labels = db.fit_predict(X_wine_scaled)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)
print(f'DBSCAN: Clusters found={n_clusters} | Noise points={n_noise}')
print(f'Label distribution: {dict(zip(*np.unique(db_labels, return_counts=True)))}')

---
### 13. PCA (Principal Component Analysis)
**Description:** Orthogonal linear transformation that projects data onto directions of maximum variance (eigenvectors of covariance matrix).

**When to use:** Dimensionality reduction before modeling, visualization (2D/3D), noise removal, multicollinearity reduction.

**Key Hyperparameters:**
- `n_components` (int or float): Number of components. Float (e.g. 0.95) = keep 95% variance.
- `whiten` (bool): Normalizes component variance — useful before SVM/KNN.

**Use Case:** Face recognition (Eigenfaces), genome dimensionality reduction.

In [ ]:
# ── PCA Example ───────────────────────────────────────────────────────────────
from sklearn.decomposition import PCA

# Keep 95% of variance
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_wine_scaled)

print(f'Original features : {X_wine_scaled.shape[1]}')
print(f'PCA components    : {pca.n_components_} (95% variance retained)')
print(f'Variance per PC   : {pca.explained_variance_ratio_.round(3)}')
print(f'Cumulative var    : {np.cumsum(pca.explained_variance_ratio_).round(3)}')

# Explained variance plot
pca_full = PCA(random_state=42).fit(X_wine_scaled)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('PCA — Explained Variance Analysis (Wine Dataset)', fontweight='bold')
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_, color='#457B9D', alpha=0.8)
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Individual Explained Variance per PC', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(pca_full.explained_variance_ratio_)+1),
             np.cumsum(pca_full.explained_variance_ratio_), 'o-', color='#E63946', linewidth=2)
axes[1].axhline(y=0.95, color='#2A9D8F', linestyle='--', label='95% threshold')
axes[1].axhline(y=0.99, color='#F4A261', linestyle='--', label='99% threshold')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_explained_variance.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Task 3 — Algorithm Selection Flowchart

In [ ]:
flowchart = """
╔══════════════════════════════════════════════════════════════════════╗
║          WEEK 6 ALGORITHM SELECTION FLOWCHART                        ║
╚══════════════════════════════════════════════════════════════════════╝

START: Do you have labels (y)?
│
├─── NO  → UNSUPERVISED LEARNING
│    │
│    ├── Do you know the number of clusters?
│    │   ├── YES → Is data convex / spherical? 
│    │   │        ├── YES → K-MEANS (fast, interpretable)
│    │   │        └── NO  → Use K-Means + tune, or Agglomerative
│    │   └── NO  → Is noise/outlier detection important?
│    │            ├── YES → DBSCAN (arbitrary shapes, noise labelling)
│    │            └── NO  → Hierarchical Clustering (dendrogram)
│    │
│    └── Do you have too many features (>50)?
│        └── YES → Apply PCA first, then cluster
│
└─── YES → SUPERVISED LEARNING
     │
     ├── Is the target continuous (regression) or categorical (classification)?
     │   └── [Assume CLASSIFICATION below]
     │
     ├── Do you have >100K samples and high-dimensional features?
     │   ├── YES → LIGHTGBM (fastest, leaf-wise boosting)
     │   └── NO  → continue ↓
     │
     ├── Do you need HIGH ACCURACY above all else?
     │   └── YES → Try XGBOOST or LIGHTGBM → then STACKING for final push
     │
     ├── Do you need INTERPRETABILITY (explain individual predictions)?
     │   ├── YES (global) → LOGISTIC REGRESSION (coefficients)
     │   │                  DECISION TREE (readable rules)
     │   └── YES (feature importance) → RANDOM FOREST (.feature_importances_)
     │
     ├── Is your dataset SMALL (<1K samples)?
     │   ├── YES + high-dim → SVM with RBF kernel (handles curse of dim)
     │   └── YES + low-dim  → KNN (simple, no training phase)
     │
     ├── Do features have COMPLEX NON-LINEAR interactions?
     │   ├── YES → RANDOM FOREST, XGBOOST, LIGHTGBM
     │   └── NO  → LOGISTIC REGRESSION (fast baseline)
     │
     ├── Are you building a ROBUST ENSEMBLE?
     │   ├── Models make DIFFERENT errors → VOTING CLASSIFIER (soft)
     │   └── Want learnable combination → STACKING CLASSIFIER
     │
     └── TOO MANY FEATURES (>50-100)? Apply PCA FIRST
         └── Keep n_components that explain 95% variance
             then re-run classification pipeline

─────────────────────────────────────────────────────────────────────
QUICK DECISION TABLE
─────────────────────────────────────────────────────────────────────
 Scenario                          →  Recommended Algorithm
─────────────────────────────────────────────────────────────────────
 Fast baseline                     →  Logistic Regression
 Need explainable rules            →  Decision Tree
 Reduce overfitting from DT        →  Random Forest
 Sequential error correction       →  AdaBoost
 High accuracy, tabular data       →  XGBoost
 Large scale, speed critical       →  LightGBM
 Diverse models, reduce variance   →  Voting Classifier
 Best possible ensemble accuracy   →  Stacking Classifier
 High-dim, small dataset           →  SVM (RBF)
 No training phase needed          →  KNN
 Cluster with known K              →  K-Means
 Cluster with outliers             →  DBSCAN
 Reduce dimensions / visualize     →  PCA
─────────────────────────────────────────────────────────────────────
"""
print(flowchart)

## Task 4 — Test 3 Algorithms on Wine Dataset

In [ ]:
# Test LR, RF, XGBoost on Wine dataset with 5-fold CV
wine = load_wine()
X_w, y_w = wine.data, wine.target
scaler_wine = StandardScaler()
X_ws = scaler_wine.fit_transform(X_w)

models = {
    'Logistic Regression':  LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':              xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, verbosity=0, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
for name, model in models.items():
    scores = cross_val_score(model, X_ws, y_w, cv=cv, scoring='accuracy')
    results.append({'Algorithm': name, 'CV Mean': scores.mean(), 'CV Std': scores.std(),
                    'Min': scores.min(), 'Max': scores.max()})

results_df = pd.DataFrame(results).sort_values('CV Mean', ascending=False)
print('=== Wine Dataset — 5-Fold CV Results ===')
print(results_df.to_string(index=False, float_format='{:.4f}'.format))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(results_df['Algorithm'], results_df['CV Mean'],
               xerr=results_df['CV Std'], color=PALETTE[:3],
               alpha=0.85, edgecolor='white', linewidth=1, capsize=5)
ax.set_xlabel('5-Fold CV Accuracy')
ax.set_title('Wine Dataset — Algorithm Comparison (5-Fold CV)', fontweight='bold')
ax.set_xlim(0.85, 1.02)
for bar, val in zip(bars, results_df['CV Mean']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('task4_wine_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

---
# PART B — Image Compression with PCA (30%)

## Tasks 5–9 — PCA Image Compression

In [ ]:
# ── Create a synthetic image (vivid, reproducible, no external download needed) ──
np.random.seed(42)
H, W = 128, 128

# Generate a structured synthetic image: gradient + circular pattern
xx, yy = np.meshgrid(np.linspace(0, 1, W), np.linspace(0, 1, H))
r_channel = (np.sin(xx * 10) * np.cos(yy * 8) * 0.5 + 0.5) * 255
g_channel = (np.sin(xx * 6 + yy * 8) * 0.5 + 0.5) * 255
b_channel = (np.cos(xx * 12) * np.sin(yy * 10) * 0.5 + 0.5) * 255

# Add some structure (circle)
cx, cy = 0.5, 0.5
dist = np.sqrt((xx - cx)**2 + (yy - cy)**2)
mask = dist < 0.3
r_channel[mask] = np.clip(r_channel[mask] + 80, 0, 255)
g_channel[mask] = np.clip(g_channel[mask] - 40, 0, 255)

img = np.stack([r_channel, g_channel, b_channel], axis=2).astype(np.uint8)
print(f'Synthetic image shape: {img.shape}  dtype: {img.dtype}')

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title('Original Image (128×128 RGB)', fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('original_image.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── PCA compression function ──────────────────────────────────────────────────
def compress_image_pca(img, n_components):
    """
    Compress an RGB image using PCA on each channel independently.
    Returns: reconstructed image (uint8), compression_ratio, mse
    """
    H, W, C = img.shape
    img_float = img.astype(np.float64)
    reconstructed_channels = []
    
    for c in range(C):  # R, G, B channels
        channel = img_float[:, :, c]   # shape: (H, W)
        pca = PCA(n_components=n_components)
        compressed = pca.fit_transform(channel)   # (H, n_components)
        reconstructed = pca.inverse_transform(compressed)  # (H, W)
        reconstructed_channels.append(reconstructed)
    
    # Stack and clip to valid range
    reconstructed = np.stack(reconstructed_channels, axis=2)
    reconstructed = np.clip(reconstructed, 0, 255).astype(np.uint8)
    
    # Compression ratio: original bytes / compressed bytes
    # Original: H * W * C float64
    # Compressed: (H * n_components + n_components * W + n_components) * C
    original_size = H * W * C
    compressed_size = C * (H * n_components + n_components * W + n_components)
    ratio = original_size / compressed_size
    
    mse = np.mean((img_float - reconstructed.astype(np.float64))**2)
    return reconstructed, ratio, mse


# ── Apply PCA with varying n_components ──────────────────────────────────────
n_components_list = [5, 20, 50, 100]
results_compression = []

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('PCA Image Compression — Effect of n_components', fontweight='bold', fontsize=14)

axes[0].imshow(img); axes[0].set_title('Original\n(128 components)', fontweight='bold'); axes[0].axis('off')

for ax, n in zip(axes[1:], n_components_list):
    rec, ratio, mse = compress_image_pca(img, n)
    results_compression.append({'n_components': n, 'Compression Ratio': ratio, 'MSE': mse,
                                 'PSNR (dB)': 10 * np.log10(255**2 / mse) if mse > 0 else np.inf})
    ax.imshow(rec)
    ax.set_title(f'n={n}\nRatio: {ratio:.2f}x\nMSE: {mse:.1f}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('pca_image_compression.png', bbox_inches='tight', dpi=150)
plt.show()

# Results table
comp_df = pd.DataFrame(results_compression)
print('\n=== PCA Compression Metrics ===')
print(comp_df.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# ── Compression metrics plots ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('PCA Image Compression — Quality vs Compression Trade-off', fontweight='bold')

axes[0].plot(comp_df['n_components'], comp_df['Compression Ratio'], 'o-', color='#E63946', linewidth=2, markersize=8)
axes[0].set_title('Compression Ratio vs n_components', fontweight='bold')
axes[0].set_xlabel('n_components'); axes[0].set_ylabel('Compression Ratio (higher = more compressed)')
axes[0].grid(True, alpha=0.3)
for x, y in zip(comp_df['n_components'], comp_df['Compression Ratio']):
    axes[0].annotate(f'{y:.1f}x', (x, y), textcoords='offset points', xytext=(0, 10), ha='center')

axes[1].plot(comp_df['n_components'], comp_df['MSE'], 's-', color='#457B9D', linewidth=2, markersize=8)
axes[1].set_title('MSE vs n_components', fontweight='bold')
axes[1].set_xlabel('n_components'); axes[1].set_ylabel('Mean Squared Error (lower = better)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(comp_df['n_components'], comp_df['PSNR (dB)'], '^-', color='#2A9D8F', linewidth=2, markersize=8)
axes[2].set_title('PSNR (dB) vs n_components', fontweight='bold')
axes[2].set_xlabel('n_components'); axes[2].set_ylabel('PSNR dB (higher = better quality)')
axes[2].axhline(y=30, color='orange', linestyle='--', label='30dB threshold (acceptable quality)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pca_compression_metrics.png', bbox_inches='tight', dpi=150)
plt.show()

---
# PART C — Interview Ready (20%)

## Q1 — Complete ML Pipeline: 1000 Samples, 200 Features

### Complete ML Pipeline: 1000 samples × 200 features → Deployed Model

---

#### Stage 1 — Data Understanding & EDA
- Check for nulls, dtypes, class imbalance, feature distributions
- 200 features → immediately flag multicollinearity and high dimensionality risk
- With 1000 samples and 200 features, **curse of dimensionality is a real concern**: distances become meaningless in high dimensions

#### Stage 2 — Preprocessing
- Handle missing values (imputation or drop)
- Encode categoricals (one-hot / ordinal)
- **StandardScaler** — mandatory before PCA, SVM, KNN, LR (all distance/magnitude sensitive algorithms)
- Train/validation/test split (70/15/15 or 80/20 with inner CV)

#### Stage 3 — Dimensionality Reduction via PCA *(Algorithm 1)*
**Why PCA here?**
- 200 features with 1000 samples → p ≈ n → high risk of overfitting
- PCA removes correlated features, reduces noise, speeds up training
- Retain `n_components` that explain **95% variance** → typically reduces from 200 to ~30–50 components

```python
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)  # fit only on train
X_test_pca  = pca.transform(X_test_scaled)        # transform test separately
```

#### Stage 4 — Baseline Model: Logistic Regression *(Algorithm 2)*
**Why LR as baseline?**
- Fast, interpretable, gives probability outputs
- After PCA, features are orthogonal → LR assumptions better satisfied
- Establishes a performance floor. If LR scores 0.9+, complex models may not be needed

```python
lr = LogisticRegression(C=1.0, max_iter=1000)
lr.fit(X_train_pca, y_train)
```

#### Stage 5 — Advanced Model: XGBoost / LightGBM *(Algorithm 3)*
**Why XGBoost/LightGBM after LR?**
- Captures non-linear interactions that LR misses
- Handles moderate datasets (1000 samples) well without overfitting (use early stopping)
- Built-in feature importance helps verify which PCA components or raw features matter
- Tune with Bayesian optimization / GridSearchCV

```python
xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                           subsample=0.8, eval_metric='logloss')
xgb_model.fit(X_train_pca, y_train, eval_set=[(X_val_pca, y_val)], early_stopping_rounds=20)
```

#### Stage 6 — Model Evaluation
- 5-fold Stratified CV on full training set
- Report: Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix
- For imbalanced classes: use ROC-AUC / PR-AUC over accuracy

#### Stage 7 — Hyperparameter Tuning
- RandomizedSearchCV (fast) or Optuna (Bayesian) for XGBoost
- Cross-validate inside the PCA pipeline to avoid data leakage

#### Stage 8 — Final Ensemble (Optional): Stacking *(Bonus Algorithm 4)*
- If single model isn't sufficient: Stack LR + XGBoost + SVM
- Meta-learner = simple LR on out-of-fold predictions

#### Stage 9 — Deployment
- Save pipeline (scaler + PCA + model) as a single `sklearn.Pipeline` object → `joblib.dump()`
- Wrap in FastAPI endpoint or Flask REST API
- Monitor: data drift, prediction confidence, concept drift

```
PIPELINE SUMMARY
─────────────────────────────────────────────────────
Raw Data (1000×200)
  → StandardScaler         [mandatory for PCA/SVM/KNN]
  → PCA (95% var, ~30-50 components) [dimensionality reduction]
  → LogisticRegression     [fast baseline]
  → XGBoost                [non-linear, high accuracy]
  → StackingClassifier     [optional final ensemble]
  → Deployed API           [joblib + FastAPI]
─────────────────────────────────────────────────────
```

## Q2 — `weekly_model_comparison()` Function

In [ ]:
def weekly_model_comparison(X, y, use_pca=False, pca_variance=0.95, cv_folds=5, random_state=42):
    """
    Trains LR, RF, XGBoost, SVM, and KNN with k-fold CV.
    Optionally applies PCA preprocessing.
    
    Parameters
    ----------
    X            : array-like, feature matrix
    y            : array-like, target labels
    use_pca      : bool, whether to apply PCA preprocessing (default False)
    pca_variance : float, explained variance threshold for PCA (default 0.95)
    cv_folds     : int, number of CV folds (default 5)
    random_state : int, for reproducibility
    
    Returns
    -------
    pd.DataFrame sorted by CV Mean Accuracy (descending)
    """
    # Preprocessing steps
    steps = [('scaler', StandardScaler())]
    if use_pca:
        steps.append(('pca', PCA(n_components=pca_variance, random_state=random_state)))
    
    # Define models
    models = {
        'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=random_state),
        'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=random_state, n_jobs=-1),
        'XGBoost':             xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, verbosity=0, random_state=random_state),
        'SVM (RBF)':           SVC(C=1.0, kernel='rbf', gamma='scale', random_state=random_state),
        'KNN (K=7)':           KNeighborsClassifier(n_neighbors=7),
    }
    
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    results = []
    
    for name, model in models.items():
        pipeline = Pipeline(steps + [('model', model)])
        scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
        results.append({
            'Algorithm':     name,
            'CV Mean Acc':   scores.mean(),
            'CV Std':        scores.std(),
            'Min':           scores.min(),
            'Max':           scores.max(),
            'PCA Applied':   use_pca
        })
    
    return pd.DataFrame(results).sort_values('CV Mean Acc', ascending=False).reset_index(drop=True)


# ── Run on Wine Dataset — with and without PCA ───────────────────────────────
wine = load_wine()

print('=== Wine Dataset — WITHOUT PCA ===')
df_no_pca = weekly_model_comparison(wine.data, wine.target, use_pca=False)
print(df_no_pca.to_string(index=False, float_format='{:.4f}'.format))

print('\n=== Wine Dataset — WITH PCA (95% variance) ===')
df_pca = weekly_model_comparison(wine.data, wine.target, use_pca=True)
print(df_pca.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ── Visualize comparison: with vs without PCA ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('weekly_model_comparison() — Wine Dataset: With vs Without PCA', fontweight='bold', fontsize=13)

for ax, df, title in zip(axes, [df_no_pca, df_pca], ['Without PCA', 'With PCA (95% variance)']):
    bars = ax.barh(df['Algorithm'][::-1], df['CV Mean Acc'][::-1],
                   xerr=df['CV Std'][::-1], color=PALETTE[:5],
                   alpha=0.85, edgecolor='white', linewidth=1, capsize=5)
    ax.set_xlabel('5-Fold CV Accuracy')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0.80, 1.05)
    for bar, val in zip(bars, df['CV Mean Acc'][::-1]):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('q2_model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## Q3 — Why Does Accuracy Drop from 0.92 → 0.85 After PCA (100→10 features)?

### Q3 Analysis: PCA Reduces 100 Features → 10 (95% variance), but Accuracy Drops 0.92 → 0.85

**This is actually a very common and nuanced scenario. Here are 5 well-analyzed reasons:**

---

#### Reason 1 — The Discarded 5% Variance Contains Class-Discriminative Information

PCA is **unsupervised** — it maximizes total variance, NOT class separability. The 5% variance thrown away (components 11–100) might not be random noise; it might contain features that strongly correlate with the target label.

**Example:** Imagine feature F100 has very low variance (rarely changes), but whenever it changes, it perfectly predicts class=1. PCA discards it as "low variance" but it's the most discriminative feature.

> **Fix:** Use **Linear Discriminant Analysis (LDA)** instead of PCA — LDA maximizes *between-class* variance. Or try **Supervised PCA** variants.

---

#### Reason 2 — The Model Overfits the 95% Threshold

"95% variance retained" was chosen arbitrarily. For THIS dataset and THIS model, the optimal number of components might be 15 or 25, not 10. The threshold 95% does not guarantee the best downstream accuracy.

> **Fix:** Treat `n_components` as a **hyperparameter** and tune it via cross-validation. Use `Pipeline` + `GridSearchCV` to search over `pca__n_components = [5, 10, 15, 20, 30]`.

```python
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
pipe = Pipeline([('scaler', StandardScaler()), ('pca', PCA()), ('clf', LogisticRegression())])
grid = GridSearchCV(pipe, {'pca__n_components': [5, 10, 15, 20, 30]}, cv=5)
grid.fit(X_train, y_train)
print('Best n_components:', grid.best_params_)
```

---

#### Reason 3 — Information Loss Through Lossy Reconstruction

The PCA transformation is a **projection** onto a subspace. Even if 95% of variance is preserved in aggregate, this is measured globally — individual samples near decision boundaries may suffer much greater local distortion. Two points from different classes that were just separable in 100D may become inseparable in 10D after projection.

> **Diagnostic:** Compute the reconstruction error per class: `||X - X_reconstructed||²`. If one class has systematically higher error, the projection is hurting that class's separability.

---

#### Reason 4 — Non-Linear Feature Interactions Are Destroyed

PCA is strictly **linear**. If the original 100 features had non-linear interactions that were encoded in combinations PCA doesn't capture (e.g., feature 5 × feature 23), those interactions are lost in the linear projection to 10 components. A non-linear model (XGBoost, SVM-RBF) that exploited these interactions in 100D is now running on a lossy linear shadow of the original data.

> **Fix:** Try **kernel PCA** (with RBF kernel) or **UMAP** for non-linear dimensionality reduction. Or use tree-based models (RF/XGBoost) directly on raw features — they handle high dimensions better than distance-based models.

---

#### Reason 5 — Data Leakage in PCA Fitting (Methodological Error)

If PCA was fitted on the **full dataset** (train + test) before the train/test split, then test-set information leaked into the principal components. This inflated the original 0.92 score. The 0.85 (correct) score reflects the true performance when PCA is fitted only on training data.

> **Fix:** Always use sklearn `Pipeline` — this guarantees PCA is only ever fit on training folds:
```python
# WRONG: PCA sees test data
X_pca = PCA(n_components=10).fit_transform(X_all)
X_tr, X_te = train_test_split(X_pca, ...)

# RIGHT: PCA only sees training data
pipeline = Pipeline([('pca', PCA(10)), ('clf', model)])
cross_val_score(pipeline, X, y, cv=5)  # PCA refitted each fold
```

---

### Summary Table

| Reason | Root Cause | Fix |
|:---|:---|:---|
| 1. Discriminative info in low-variance components | PCA is unsupervised | Use LDA or Supervised PCA |
| 2. Wrong n_components (95% ≠ optimal) | Threshold chosen arbitrarily | Tune via CV |
| 3. Decision boundary distortion | Lossy projection in 10D | Check per-class reconstruction error |
| 4. Non-linear interactions lost | PCA is linear | Kernel PCA, UMAP, or skip PCA for trees |
| 5. Data leakage in original score | PCA fit on full data | Always use Pipeline |


---
# PART D — AI-Augmented Task: Week 6 Study Guide (10%)

## Tasks 10–13 — Comprehensive Week 6 Study Guide

---

# 📚 WEEK 6 STUDY GUIDE — ML & AI
### IIT Gandhinagar | PG Diploma | Saturday Assessment Prep

---

## MODULE 1: SUPERVISED LEARNING — CLASSICAL

### Logistic Regression
- **Core concept:** $\hat{y} = \sigma(W^T x + b)$ where $\sigma(z) = \frac{1}{1+e^{-z}}$
- **Loss:** Binary Cross-Entropy; **Regularization:** L1 (Lasso, sparse), L2 (Ridge, shrinks)
- **Key fact:** Linear decision boundaries. `C` = 1/λ (inverse regularization)
- **Interview Q:** *How is LR different from Linear Regression?* → LR outputs probabilities via sigmoid; uses cross-entropy loss, not MSE
- **Code pattern:** `LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)`

### Decision Tree
- **Core concept:** Greedy top-down splitting by information gain (entropy) or Gini impurity
- **Gini:** $1 - \sum p_i^2$ | **Entropy:** $-\sum p_i \log_2 p_i$
- **Key fact:** No scaling needed. Prone to overfitting → control with `max_depth`, `min_samples_split`
- **Interview Q:** *Why do DTs overfit?* → Deep trees memorize training noise (high variance)
- **Code pattern:** `DecisionTreeClassifier(max_depth=5, min_samples_leaf=4)`

---

## MODULE 2: ENSEMBLE METHODS

### Random Forest
- **Core concept:** Bagging (bootstrap) + feature randomness → decorrelated trees
- **Key fact:** `n_estimators` ↑ reduces variance but ∞ trees give diminishing returns
- **Interview Q:** *Why is RF better than a single DT?* → Averaging reduces variance; diverse trees reduce overfitting
- **Code pattern:** `RandomForestClassifier(n_estimators=200, max_features='sqrt')`

### AdaBoost
- **Core concept:** Sequential boosting — each iteration upweights misclassified points
- **Key fact:** Final prediction = weighted sum of weak learners. Sensitive to noise/outliers
- **Interview Q:** *AdaBoost vs RF?* → RF: parallel (bagging), reduces variance; AdaBoost: sequential (boosting), reduces bias
- **Code pattern:** `AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=100)`

### XGBoost
- **Core concept:** Gradient boosting + L1/L2 regularization + 2nd-order gradients + column block parallel
- **Key hyperparams:** `n_estimators`, `learning_rate`, `max_depth`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`
- **Interview Q:** *XGBoost vs GradientBoosting?* → XGB adds regularization, handles missing values, parallel column sampling
- **Code pattern:** `XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, subsample=0.8)`

### LightGBM
- **Core concept:** **Leaf-wise** growth (vs level-wise in XGBoost) + histogram-based splitting + GOSS + EFB
- **Key fact:** Much faster on large data; `num_leaves` is the key complexity parameter
- **Interview Q:** *LightGBM vs XGBoost?* → LightGBM: faster (histogram, GOSS), can overfit on small data; XGBoost: slower, more robust
- **Code pattern:** `LGBMClassifier(num_leaves=31, min_child_samples=20, learning_rate=0.05)`

### Voting Classifier
- **Core concept:** Hard voting = majority class; Soft voting = average probabilities (better)
- **Key fact:** Works best when base models are **diverse** (different algorithms, not just different params)
- **Interview Q:** *When is Voting better than a single model?* → When individual models make different mistakes
- **Code pattern:** `VotingClassifier(estimators=[('lr', lr), ('rf', rf), ('svm', svm)], voting='soft')`

### Stacking
- **Core concept:** Base learners generate out-of-fold predictions → meta-learner trains on those predictions
- **Key fact:** Must use `cross_val_predict` (OOF) to avoid data leakage in meta-features
- **Interview Q:** *Stacking vs Voting?* → Stacking learns HOW to combine; Voting uses fixed rule (majority/average)
- **Code pattern:** `StackingClassifier(estimators=[...], final_estimator=LogisticRegression(), cv=5)`

---

## MODULE 3: KERNEL & DISTANCE-BASED

### SVM
- **Core concept:** Maximize margin = $\frac{2}{||w||}$ | Support vectors define the boundary
- **Kernel trick:** $K(x_i, x_j) = \phi(x_i)^T \phi(x_j)$ — maps to higher dimension without explicit computation
- **Key fact:** Scale features! SVM is sensitive to magnitude. C controls margin vs misclassification
- **Interview Q:** *What are support vectors?* → The training points closest to the decision boundary; removing others doesn't change the model
- **Code pattern:** `SVC(C=1.0, kernel='rbf', gamma='scale')`

### KNN
- **Core concept:** Lazy learner — all computation at prediction time. No explicit model
- **Key fact:** Scale features! Uses Euclidean distance by default. High K = smoother boundary (less overfit)
- **Time complexity:** Prediction O(n·d) — slow for large datasets
- **Interview Q:** *KNN disadvantages?* → No training phase but slow prediction, sensitive to irrelevant features, needs scaling
- **Code pattern:** `KNeighborsClassifier(n_neighbors=7, metric='euclidean')`

---

## MODULE 4: UNSUPERVISED LEARNING

### K-Means
- **Core concept:** Minimize WCSS = $\sum_{k} \sum_{x \in C_k} ||x - \mu_k||^2$ | EM-style iterative optimization
- **Limitations:** Assumes spherical clusters, sensitive to initialization, must specify K
- **Choose K:** Elbow method (plot WCSS vs K) | Silhouette score (higher = better separation)
- **Interview Q:** *K-Means is 'greedy' — what does that mean?* → Makes locally optimal decisions at each step; can get stuck in local minima
- **Interview Q:** *How does KMeans++ help?* → Spreads initial centroids probabilistically; reduces local minima risk

### DBSCAN
- **Core concept:** Core point (≥ min_samples neighbours within eps) → expand cluster; isolated = noise
- **Point types:** Core, Border (reachable but not core), Noise (outlier)
- **Tune eps:** k-distance plot (sort distances to k-th nearest neighbour; find elbow)
- **Interview Q:** *K-Means vs DBSCAN?* → K-Means: fixed K, spherical; DBSCAN: arbitrary shapes, finds outliers, no K needed

---

## MODULE 5: DIMENSIONALITY REDUCTION — PCA

### PCA — Key Concepts
- **Core math:** Eigenvectors of covariance matrix $\Sigma = \frac{1}{n}X^TX$ → ordered by eigenvalue (variance)
- **Steps:** Center data → compute covariance matrix → eigendecomposition → project onto top k eigenvectors
- **Explained variance ratio:** $\text{EVR}_i = \frac{\lambda_i}{\sum \lambda_j}$ — fraction of variance explained by PC $i$
- **CRITICAL:** Fit PCA only on training data, transform test data separately (avoid data leakage)

### PCA — Interview Questions & Answers
1. **Q: What does PC1 represent?** → The direction of maximum variance in the data
2. **Q: Are PCs correlated?** → No. PCs are orthogonal by construction
3. **Q: PCA vs LDA?** → PCA: unsupervised, maximizes variance; LDA: supervised, maximizes class separability
4. **Q: When is PCA harmful?** → When discriminative info is in low-variance directions; when features are non-linear
5. **Q: Why scale before PCA?** → Without scaling, high-variance features dominate PCs regardless of relevance

---

## MODULE 6: QUICK FORMULA SHEET

| Metric | Formula |
|:---|:---|
| Silhouette score | $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$ |
| ARI | Adjusted for chance; range [-1, 1]; 1 = perfect |
| Gini impurity | $1 - \sum_k p_k^2$ |
| Information gain | $H(parent) - \sum_j \frac{n_j}{n} H(child_j)$ |
| SVM margin | $\frac{2}{||w||}$ |
| PCA variance | $\text{EVR}_i = \lambda_i / \sum \lambda_j$ |

---

## MODULE 7: WHEN TO SCALE?

| Algorithm | Needs Scaling? | Reason |
|:---:|:---:|:---|
| Logistic Regression | ✅ Yes | Gradient descent converges faster |
| SVM | ✅ Yes | Kernel distances are magnitude-sensitive |
| KNN | ✅ Yes | Euclidean distance is magnitude-sensitive |
| PCA | ✅ Yes | Covariance matrix dominated by high-variance features |
| Decision Tree | ❌ No | Splits are threshold-based, not distance-based |
| Random Forest | ❌ No | Tree-based, scale-invariant |
| XGBoost | ❌ No | Tree-based, scale-invariant |
| LightGBM | ❌ No | Histogram-based, scale-invariant |
| AdaBoost | ❌ No | Tree stumps are scale-invariant |

---

## MODULE 8: COMMON EXAM/INTERVIEW TRAPS

1. **PCA data leakage:** Never fit PCA on full dataset before split. Always use Pipeline.
2. **K-Means initialization:** Always use `init='k-means++'` and `n_init≥10`.
3. **DBSCAN noise label:** Noise points get label=-1 in sklearn. Don't mistake for cluster 1.
4. **Stacking leakage:** Meta-features must come from out-of-fold predictions, not in-fold.
5. **Soft voting:** Requires `probability=True` for SVC.
6. **Silhouette range:** [-1, 1]. Negative = point probably in wrong cluster.
7. **LightGBM leaf-wise:** Can overfit on small data → always set `min_child_samples`.
8. **ARI vs NMI:** ARI is adjusted for chance (can be negative); NMI is always [0,1].

---
*Study Guide generated and verified for D34 PM Assessment Prep*  
*IIT Gandhinagar | PG Diploma in AI-ML & Agentic AI Engineering*

---
# Final Summary Dashboard

In [ ]:
print('╔══════════════════════════════════════════════════════════════════╗')
print('║         D34 PM — COMPLETE ASSIGNMENT SUMMARY                     ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print('║ PART A: 13 Algorithms covered with code, hyperparams, use cases  ║')
print('║   Algorithm Selection Flowchart: text-based + decision table      ║')
print('║   Wine Dataset: LR, RF, XGBoost tested with 5-fold CV            ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print('║ PART B: PCA Image Compression                                     ║')
print('║   n_components: [5, 20, 50, 100] on 128x128 RGB image            ║')
print('║   Metrics: Compression Ratio, MSE, PSNR reported per config      ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print('║ PART C: Interview Ready                                           ║')
print('║   Q1: Full ML pipeline (1000 samples, 200 features)              ║')
print('║   Q2: weekly_model_comparison() with PCA option                  ║')
print('║   Q3: 5 reasons accuracy drops after PCA (with fixes)            ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print('║ PART D: Comprehensive Week 6 Study Guide                          ║')
print('║   8 modules: algorithms, formulas, scaling guide, exam traps     ║')
print('╚══════════════════════════════════════════════════════════════════╝')